<a href="https://www.kaggle.com/code/afiaanimahfosu/selah-notebook?scriptVersionId=335844587" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# Selah — Scripture for the Moments Between Everything

A Progressive Web App that connects to Google Calendar and delivers Scripture 
and an AI-generated reflection the moment a class or meeting ends.

**Live demo:** https://selah-five-mu.vercel.app
**Full source code:** https://github.com/efyastar/Selah
**Video demo:** [add your YouTube link once uploaded]

This notebook walks through the key engineering pieces of Selah's frontend 
and backend — the calendar polling with OAuth token refresh, the dual verse 
modes, and the notification system — as verification of the working code 
behind the demo.

In [ ]:
@app.get("/verse")
def get_verse(day: int = None, book: str = None, chapter: int = 1, start: int = 1, count: int = 1):
    from datetime import datetime

    headers = {
        "X-YVP-App-Key": YOUVERSION_APP_KEY,
        "Accept": "application/json"
    }

    if book:
        if count > 1:
            end = start + count - 1
            passage_id = f"{book}.{chapter}.{start}-{end}"
        else:
            passage_id = f"{book}.{chapter}.{start}"
    else:
        day_of_year = day if day else datetime.now().timetuple().tm_yday
        votd_response = requests.get(
            f"https://api.youversion.com/v1/verse_of_the_days/{day_of_year}",
            headers=headers
        )
        votd_data = votd_response.json()
        passage_id = votd_data.get("passage_id") or votd_data.get("data", [{}])[0].get("passage_id")

    verse_response = requests.get(
        f"https://api.youversion.com/v1/bibles/3034/passages/{passage_id}",
        headers=headers
    )
    return verse_response.json()

In [ ]:
@app.get("/reflection")
def get_reflection(event: str = "class"):
    token = get_gloo_token()
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "gloo-openai-gpt-5-mini",
        "instructions": "You are a faith-based companion. Given what someone just finished, suggest a short warm one-sentence reflection to pair with Scripture. Under 20 words.",
        "input": [
            {"role": "user", "content": f"I just finished: {event}"}
        ]
    }
    response = requests.post(
        "https://platform.ai.gloo.com/ai/v1/responses",
        headers=headers,
        json=payload
    )
    data = response.json()
    output = data["output"]
    message = next(item for item in output if item["type"] == "message")
    return {"reflection": message["content"][0]["text"]}

In [ ]:
@app.get("/calendar/check")
def check_calendar(access_token: str, refresh_token: str = None):
    from datetime import datetime, timezone, timedelta
    now = datetime.now(timezone.utc)
    window_start = now - timedelta(minutes=5)

    def fetch_events(token):
        headers = {"Authorization": f"Bearer {token}"}
        return requests.get(
            "https://www.googleapis.com/calendar/v3/calendars/primary/events",
            headers=headers,
            params={
                "maxResults": 10,
                "orderBy": "startTime",
                "singleEvents": True,
                "timeMin": (now - timedelta(hours=12)).isoformat(),
                "timeMax": (now + timedelta(hours=12)).isoformat()
            }
        )

    response = fetch_events(access_token)

    if response.status_code == 401 and refresh_token:
        new_token = refresh_access_token(refresh_token)
        if new_token:
            response = fetch_events(new_token)
            access_token = new_token

    data = response.json()
    events = data.get("items", [])
    for event in events:
        end_time = event.get("end", {}).get("dateTime")
        if end_time:
            end_dt = datetime.fromisoformat(end_time)
            if window_start <= end_dt <= now:
                return {
                    "event_ended": True,
                    "event_name": event.get("summary", "your session"),
                    "new_access_token": access_token
                }
    return {"event_ended": False, "new_access_token": access_token}

In [ ]:
useEffect(() => {
  if (!accessToken) return;
  const interval = setInterval(() => {
    const refreshToken = localStorage.getItem('selah_refresh');
    fetch(`${API}/calendar/check?access_token=${accessToken}&refresh_token=${refreshToken || ''}`)
      .then(res => res.json())
      .then(data => {
        if (data.new_access_token && data.new_access_token !== accessToken) {
          setAccessToken(data.new_access_token);
          localStorage.setItem('selah_token', data.new_access_token);
        }
        if (data.event_ended) {
          const eventKey = `notified_${data.event_name}`;
          const alreadyNotified = localStorage.getItem(eventKey);
          const lastNotifiedTime = alreadyNotified ? parseInt(alreadyNotified) : 0;
          const now = Date.now();

          if (now - lastNotifiedTime > 30 * 60 * 1000) {
            setCurrentEvent(data.event_name);
            setEventJustEnded(true);
            sendSelahNotification(data.event_name);
            localStorage.setItem(eventKey, now.toString());
          }
        }
      });
  }, 60000);
  return () => clearInterval(interval);
}, [accessToken]);

In [ ]:
useEffect(() => {
  if (showSelah) {
    const verseUrl = mode === 'journey'
      ? `${API}/verse?book=${journeyBook}&chapter=${journeyChapter}&start=${journeyVerse}&count=${versesPerMoment}`
      : `${API}/verse`;

    fetch(verseUrl)
      .then(res => res.json())
      .then(data => setVerse(data));

    fetch(`${API}/reflection?event=${currentEvent}`)
      .then(res => res.json())
      .then(data => {
        setReflection(data.reflection);
      })
      .catch(err => console.log("REFLECTION ERROR:", err));
  }
}, [showSelah]);

In [ ]:
const sendSelahNotification = (eventName) => {
  if (!('Notification' in window)) return;

  if (Notification.permission === 'granted') {
    const notification = new Notification('Take a Selah moment', {
      body: `${eventName} just ended. Pause, breathe, and reflect.`,
      icon: '/logo192.png',
      tag: `selah-moment-${Date.now()}`,
      requireInteraction: true,
    });

    notification.onclick = () => {
      window.focus();
      setShowSelah(true);
    };
  }
};

In [ ]:
const closeSelah = () => {
  setShowSelah(false);
  setVerse(null);
  setReflection(null);
  setEventJustEnded(false);
  if (mode === 'journey') {
    const nextVerse = journeyVerse + versesPerMoment;
    setJourneyVerse(nextVerse);
    localStorage.setItem('selah_verse', nextVerse);
  }
  setCurrentVideo(videos[Math.floor(Math.random() * videos.length)]);
  setCurrentMusic(musicTracks[Math.floor(Math.random() * musicTracks.length)]);
};